<a href="https://colab.research.google.com/github/flintop123/Bits-AI-Demo/blob/main/pdfhintoeng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import argparse
import re
import sys

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-hi-en"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def split_sentences(text: str) -> list[str]:
    """Split text at sentence endings and newlines."""
    parts = re.split(r"(?<=[।.?!])\s+|\n+", text)
    return [part.strip() for part in parts if part.strip()]

In [2]:
pip install pypdf

In [4]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract plain text from all pages of a PDF file."""
    reader = PdfReader(pdf_path)
    extracted_pages = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            extracted_pages.append(page_text)
    return "\n".join(extracted_pages)

In [23]:
from pathlib import Path
from pypdf import PdfReader

# 1. Use a raw string (r"...") so backslashes aren't treated as escape characters
pdf_path = Path(r"/content/apeda_makhana_report_hindi-1-5.pdf")

def extract_pdf(path: Path) -> str:
    # Verify existence with pathlib
    if not path.is_file():
        raise FileNotFoundError(f"Python cannot see the file at: {path}")

    reader = PdfReader(path)
    text_content = []
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text_content.append(extracted)

    return "\n".join(text_content)

# Run extraction
try:
    text = extract_pdf(pdf_path)
    print(f"Successfully extracted {len(text)} characters from PDF.")
    print(text)
except Exception as e:
    print(f"Error: {e}")

Successfully extracted 2170 characters from PDF.
भारत से मखाना ननरायात को प्ोतसानित नकए जाने िेतु कारयानीनत
भारत से मखाना ननरायात को प्ोतसानित  
नकए जाने िेतु कारयानीनत
 
िरया वरयान
तनर संतवाल
अनिनत बंसल
लक्मीकांत
अशोक गुलाटी
www.academicfoundation.or g
INDIA

2025 में सर्वप्रथम प्रकाशित 
�ारा
एएफ प्रैस
नं. 35, सेक्टर 7, 
आईएम्टी मानेसर, 
गुरुग्ाम –122050
हररयाणा (भारत) 
फोनः +91-124-4215070 / 71 
ईमेल: booksaf@gmail.com 
www.academicfoundation.org
सहयोग में
अंतरराष्ट्ीर आन्याक संबंरों पर भारतीर अनुसंरान परररि (आईसीआरआईईआर)
कोर 6-ए, चौथी मंश़िल, इंशिया हैशि्टै्ट सें्टर,
लोधी रोि, नई शिलली – 110003।
ईिर भरन, पललॉ्ट संखया 16-17, पुषप शरहार, संसथागत क्ेत्र,
सेक्टर 6, नई शिलली – 110017।
िूरभाष: 91-11-43112400
रेिसाइ्ट: www.icrier.org
सरा्वशधकार सुरशक्त
इस पुसतक का कोई भी भाग शिना पूर्व शलशित अनुमशत क े , चाहे रह कलॉपीराइ्ट धारक हो या प्रकािक, शकसी भी रूप में — इलेकट्लॉशनक, यांशत्रक, 
फो्टोकलॉपी, ररकलॉशििंग या अनय शकसी माधयम से — पुनरुतपाशित, संग्शहत या प्रसाररत नहीं शकया जा सकता।
असरीकरण:


In [8]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "Helsinki-NLP/opus-mt-hi-en"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


... (2 outputs left out)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  304MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [21]:
def translates(text: str, batch_size: int = 8) -> str:
    """Translate English text into Hindi in batches."""
    if not text or not str(text).strip():
        return ""

    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")

    sentences = split_sentences(text)
    results = []

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start : start + batch_size]
        print(batch)

        inputs = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt",
        ).to(DEVICE)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                num_beams=4,
                #max_new_tokens=256,
            )

        translated = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True,
        )
        #print(translated)
        results.extend(translated)

    # Ensure every element is cast to string before calling .strip()
    return " ".join(str(res).strip() for res in results if res is not None)

In [24]:
# Run extraction
try:
    text_content = extract_pdf(pdf_path)
    # englisth_content = translates("अनाज या अनाज उतपािों को फु लाकर या भूनकर प्राप्त िाद्य पिाथ्व", 8)
    # print(f"Successfully extracted {len(text)} characters from PDF.")
    englisth_content = translates(text_content, 8)
    print(englisth_content)
except Exception as e:
    print(f"Error: {e}")


['भारत से मखाना ननरायात को प्ोतसानित नकए जाने िेतु कारयानीनत', 'भारत से मखाना ननरायात को प्ोतसानित', 'नकए जाने िेतु कारयानीनत', 'िरया वरयान', 'तनर संतवाल', 'अनिनत बंसल', 'लक्मीकांत', 'अशोक गुलाटी']
['www.academicfoundation.or g', 'INDIA', '2025 में सर्वप्रथम प्रकाशित', '�ारा', 'एएफ प्रैस', 'नं.', '35, सेक्टर 7,', 'आईएम्टी मानेसर,']
['गुरुग्ाम –122050', 'हररयाणा (भारत)', 'फोनः +91-124-4215070 / 71', 'ईमेल: booksaf@gmail.com', 'www.academicfoundation.org', 'सहयोग में', 'अंतरराष्ट्ीर आन्याक संबंरों पर भारतीर अनुसंरान परररि (आईसीआरआईईआर)', 'कोर 6-ए, चौथी मंश़िल, इंशिया हैशि्टै्ट सें्टर,']
['लोधी रोि, नई शिलली – 110003।', 'ईिर भरन, पललॉ्ट संखया 16-17, पुषप शरहार, संसथागत क्ेत्र,', 'सेक्टर 6, नई शिलली – 110017।', 'िूरभाष: 91-11-43112400', 'रेिसाइ्ट: www.icrier.org', 'सरा्वशधकार सुरशक्त', 'इस पुसतक का कोई भी भाग शिना पूर्व शलशित अनुमशत क े , चाहे रह कलॉपीराइ्ट धारक हो या प्रकािक, शकसी भी रूप में — इलेकट्लॉशनक, यांशत्रक,', 'फो्टोकलॉपी, ररकलॉशििंग या अनय शकसी माधयम से — पुनरुतपाशित, संग्शहत या 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')